# All-atom PhantomWalk: initializing a dense polymer melt

This tutorial builds a dense all-atom polyethylene melt at its target density on a CPU:

1. **Build chains** with a `Polymer` (here `PolyEthylene`).
2. **Place them at the target density** with `AtomisticRandomWalk`, turning every rotatable bond with `LocalBoltzmannSampler`. Chains overlap; that is intended.
3. **Write an all-atom DPD force field**: bonded terms from a standard force field, scaled up, plus a soft DPD pair force that lets atoms pass through each other. FlowerMD has three ways to get the bonded terms, all shown here:
   - `UFF_DPD`: UFF, from RDKit.
   - `OpenFF_DPD`: a SMIRNOFF force field such as Sage, through OpenFF Interchange.
   - `XML_DPD`: any FlowerMD XML force field, such as `OPLS_AA` or your own file.
4. **Relax** with `PhantomWalk.run_initialization`: DPD until the energies are stationary, then FIRE.
5. **Hand off** the coordinates to a standard force field for minimization.

The result is *minimizer-ready coordinates at the right density*, not an equilibrated melt. Units are kJ/mol, nm and amu. Requires RDKit and OpenFF (openff-toolkit and openff-interchange).

In [ ]:
import warnings

import numpy as np
import unyt as u

from flowermd.library import (
    OPLS_AA,
    UFF_DPD,
    XML_DPD,
    AtomisticRandomWalk,
    LocalBoltzmannSampler,
    OpenFF_DPD,
    PhantomWalk,
    PolyEthylene,
)

warnings.filterwarnings("ignore")

## 1-2. Chains placed at the target density

Ten 20-mer chains (about 1,200 atoms). `AtomisticRandomWalk` sizes a cubic box for 0.78 g/cm³ (a polyethylene melt at 413 K) and gives each chain a random start and orientation. `LocalBoltzmannSampler` first relaxes the chain under the Sage force field's bonded terms, then turns every rotatable C-C bond to a torsion drawn from local Boltzmann weights at 413 K: Sage's torsion about the bond plus the Lennard-Jones energy of atoms up to four bonds apart, with no heavy atoms closer than 0.30 nm. This gives realistic chain dimensions; bond lengths, bond angles and any stereocenters keep the relaxed geometry.

Writing a DPD force field renames the system's particles to that force field's atom types, so each force field below gets its own system. `place` builds the same melt each time (same `seed`).

In [ ]:
def place():
    chains = PolyEthylene(lengths=20, num_mols=10)
    return AtomisticRandomWalk(
        chains,
        density=0.78 * u.g / u.cm**3,
        torsion_sampler=LocalBoltzmannSampler(temperature=413),
        seed=1,
    )


system = place()
print("atoms:", system.n_particles, " box (nm):", system.box.Lx)

## 3. The all-atom DPD force field

Each DPD force field writes a GMSO XML file for its system and renames the particles to the refined atom types. They share the defaults of the PhantomWalk protocol: bonded terms ×30 (torsions too, `bonded_scale` and `dihedral_scale`), DPD repulsion `A = 52300` kJ/(mol nm) and friction `gamma = 4091` amu/ps, each weighted per pair by the Lennard-Jones well depths of the source force field, and a 0.35 nm cutoff. Only the source of the bonded terms and well depths differs. Applying a DPD force field also restrains every tetrahedral stereocenter, so DPD cannot invert it (`AtomisticRandomWalk(protect_stereo=False)` turns this off); polyethylene has none.

### 3a. UFF (`UFF_DPD`)

RDKit's UFF covers almost any organic molecule and needs nothing beyond RDKit.

In [ ]:
ff = UFF_DPD(system, file_path="pe_uff_dpd.xml")
system.apply_forcefield(r_cut=0.35, force_field=ff, kT=4.184)
systems = {"UFF": system}
print("types:", system.hoomd_snapshot.particles.types)

### 3b. A SMIRNOFF force field (`OpenFF_DPD`)

OpenFF Interchange assigns the terms from a SMIRNOFF force field, Sage 2.3.0 by default (`force_field` takes any name or path that `openff.toolkit.ForceField` accepts). Its impropers are kept, summed into one term per center.

In [ ]:
system_openff = place()
ff = OpenFF_DPD(
    system_openff,
    file_path="pe_openff_dpd.xml",
    force_field="openff-2.3.0.offxml",
)
system_openff.apply_forcefield(r_cut=0.35, force_field=ff, kT=4.184)
systems["OpenFF"] = system_openff
print("types:", system_openff.hoomd_snapshot.particles.types)

### 3c. Any FlowerMD XML force field (`XML_DPD`)

`XML_DPD` takes the bonds, angles, dihedrals and Lennard-Jones well depths from any FlowerMD XML force field, typed with GMSO as `System` types it. Here that is `OPLS_AA`; for your own file use `FF_from_file("my_ff.xml")`. The force field's Lennard-Jones and Coulomb terms are not used: the well depths only weight the DPD pair force, and charges are dropped. Impropers are not typed, and Ryckaert-Bellemans and OPLS dihedrals are rewritten as periodic ones. Atom types with a zero well depth (OPLS-AA's hydrogens on O, N and S) are weighted as if they had the smallest nonzero one, with a warning.

For other parameter sources, subclass `flowermd.base.BaseAllAtomDPDForcefield` and implement `_terms` (see the all-atom PhantomWalk page of the docs).

In [ ]:
system_xml = place()
ff = XML_DPD(system_xml, force_field=OPLS_AA(), file_path="pe_xml_dpd.xml")
system_xml.apply_forcefield(r_cut=0.35, force_field=ff, kT=4.184)
systems["XML (OPLS-AA)"] = system_xml
print("types:", system_xml.hoomd_snapshot.particles.types)

## 4. DPD until stationary, then FIRE

`run_initialization` runs 3,500 DPD steps, then 500-step chunks (five energy samples each) until every force's per-particle energy changes by less than 2 % for two chunks in a row, capped at `n_steps_dpd`. FIRE then runs with the conservative part of the DPD force. If the cap is reached the coordinates are still returned with `dpd_converged = False` and a warning (`require_convergence=True` makes it an error).

Each of the three systems is relaxed the same way.

In [ ]:
sims = {}
for name, s in systems.items():
    sim = PhantomWalk(
        s.hoomd_snapshot,
        s.hoomd_forcefield,
        n_steps_dpd=40000,
        n_steps_fire=100,
        dt=2e-4,
        gsd_write_freq=1e5,
        log_write_freq=1e5,
        run_now=False,
    )
    tag = name.split()[0].lower()
    record = sim.run_initialization(record_file=f"pe_{tag}_record.json")
    sims[name] = sim
    print(
        f"{name:14s} DPD steps: {record['dpd_steps']:6d}"
        f"  converged: {record['dpd_converged']}"
        f"  ring piercings: {record['ring_piercings']}"
    )

## 5. Hand-off

The final unwrapped coordinates (nm) go back into the mBuild compound for a parameterizer such as OpenFF Interchange or foyer; `sim.save_restart_gsd()` keeps the HOOMD state instead. Here we save the melt relaxed with UFF.

In [ ]:
snap = sims["UFF"].state.get_snapshot()
system.system.xyz = (
    snap.particles.position + snap.particles.image * snap.configuration.box[:3]
)
system.system.save("pe_melt.mol2", overwrite=True)